In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.utils.plotting import plot_simulation_dashboard, plot_benchmarker_results
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import (
    build_approach,
    AugmentedFCLockedControl,
    AugmentedPolicyControl,
    AugmentedValueControl,
)

In [ ]:
# 1. Setup the Environment and Lock the Physics/Grid
env = EnvConfig()

# We choose a "sensible" baseline grid that runs fast but has decent resolution.
# We will NOT change these parameters in this specific notebook.
config = SimConfig(
    dP=200.0,
    Dt=300,                 
    N_Pd=6,                 
    use_smart_grid=True,    
    n_pack=2,
)

In [ ]:
# 2. Load the Fleet Data into RAM
fleet_data = load_and_cache_entire_fleet(env)

# Leave empty to run the full, rigorous 11-day LOOCV for the paper.
exclude_days = [1,2,3] 
benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)

In [ ]:
# 3. Define the Control Architectures
# We are testing the continuous "Real" physics tracking.
approaches = {
    "FCLocked_Real": build_approach(
        controller_cls=AugmentedFCLockedControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    ),
    "Policy_Real": build_approach(
        controller_cls=AugmentedPolicyControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    ),
    "Value_Real": build_approach(
        controller_cls=AugmentedValueControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    ),
}

In [ ]:
# 4. Run the Benchmarks
print("--- RUNNING ARCHITECTURE SHOWDOWN (LOOCV) ---")
reports = {}

for app_name, factory in approaches.items():
    print(f"\nEvaluating {app_name}...")
    reports[app_name] = benchmarker.run_leave_one_out(factory)

# Extract and combine the 'Average' row from each approach's summary
summary_rows = {name: rep.summary.loc['Average'] for name, rep in reports.items()}
master_summary = pd.DataFrame(summary_rows).T

print("\n--- MASTER PERFORMANCE SUMMARY (AVERAGE OVER ALL DAYS) ---")
print_markdown_table(master_summary)

In [ ]:
# 5. Generate Figures
# Plot 1: The overarching bar chart proving Value Control wins
plot_benchmarker_results(
    master_summary, 
    title="Architecture Showdown: Average Realized Costs", 
    plot_type='bar',
    save_plot=False # Set to True when you want to save for the paper
)

# Plot 2: Deep-dive into a specific day to show WHY Value Control wins
test_day = 9
run_id = f"Day {test_day}"

print(f"\n--- GENERATING DASHBOARDS FOR DAY {test_day} ---")
for app_name in approaches.keys():
    df_telemetry = reports[app_name].get_telemetry(run_id)
    plot_simulation_dashboard(
        df_telemetry, 
        config, 
        title=f"Day {test_day} Execution: {app_name}", 
        indiv=False,
        save_plot=False
    )